# Final paired AR order-selection experiment

Use the frozen 250-series subset: 50 series at each true order p=1,...,5,
pooled across the five Beta innovation shapes. All series have T=1000.
Compare **empirical minimum width + descent chain**, **BIC**, and **PACF cutoff**.

The resulting matrix follows the order-selection table in the thesis: each cell
is **correct/50 (under/over)**. This notebook saves new results outside the
protected thesis directory; it does not edit that table.


In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "Code/Code_AR_processes/ar_tools/final_order_selection.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from Code.Code_AR_processes.ar_tools import final_order_selection as study
study = importlib.reload(study)
AR_DIR = PROJECT_ROOT / "Code/Code_AR_processes"


## Frozen design and optimizer settings

The dataset is [030cde6f724e3c30](../inputs/ar_order_selection_final/030cde6f724e3c30/README.md).
No series is simulated or resampled here. Only the saved observations enter the
three selectors; true orders enter the scoring after selection.

Use K=10 and the tuned coefficient-study settings: initial step 0.1/L, where
L is the Euclidean diameter of the current cloud, half-life 100, at most 2,000
evaluated states, patience 200, delta 1e-8, and strict width improvement.
There is one padded previous-dimension start per k>=2, no random multistart,
and no stationarity checks or restarts. k=1 is its observed range.

All geometric dimensions use the same endpoints t=10,...,1000. Saved curves
are the actual fitted widths, without smoothing or a cumulative-minimum repair.
These are feasible numerical fits, not certified global minimum widths.

BIC uses the established intercept-including conditional least-squares helper,
and PACF uses the established Yule-Walker helper and cutoff 1.96/sqrt(T).
Their candidates are orders 0,...,9. The unchanged descent-chain selector uses
orders 1,...,8 with K=10, as in its shared implementation.

Sources: [final coefficient-study settings](../ar_tools/final_coefficient_estimation.py),
[shared descent](../ar_tools/descent.py), [classical selectors](../ar_tools/classical_selectors.py),
and [descent-chain selector](../ar_tools/breakpoint_selectors.py), function `descent_chain`
(Definitions 1.2.8-1.2.10 and Eq. (1.6) of the supplied preview, pp. 5-6).


In [2]:
DATASET_DIRECTORY = AR_DIR / "inputs/ar_order_selection_final/030cde6f724e3c30"
OUTPUT_ROOT = AR_DIR / "outputs/csv/ar_order_selection_final"
CONFIGURATION = study.default_configuration()
CONFIGURATION.update(K=10, maximum_classical_order=9, scale=0.1,
                     half_life=100.0, iterations=2000, patience=200,
                     delta=1e-8, improvement_tolerance=0.0)
N_JOBS = 4
display(pd.Series(CONFIGURATION, name="Setting"))


K                             1.000000e+01
maximum_classical_order       9.000000e+00
scale                         1.000000e-01
half_life                     1.000000e+02
iterations                    2.000000e+03
patience                      2.000000e+02
delta                         1.000000e-08
improvement_tolerance         0.000000e+00
optimizer_seed                2.026102e+09
pacf_confidence_multiplier    1.960000e+00
Name: Setting, dtype: float64

In [3]:
run = study.run_order_selection(DATASET_DIRECTORY, OUTPUT_ROOT,
                                CONFIGURATION, n_jobs=N_JOBS)
results = run["results"]
assert len(results) == 250 * len(study.METHODS)
assert results.groupby(["method", "order"]).size().eq(50).all()
print("Saved results:", run["directory"])


Completed 10/250 series (2.2s)
Completed 20/250 series (2.3s)
Completed 30/250 series (2.3s)
Completed 40/250 series (2.3s)
Completed 50/250 series (2.3s)
Completed 60/250 series (2.3s)
Completed 70/250 series (2.3s)
Completed 80/250 series (2.3s)
Completed 90/250 series (2.3s)
Completed 100/250 series (2.3s)
Completed 110/250 series (2.3s)
Completed 120/250 series (2.4s)
Completed 130/250 series (2.4s)
Completed 140/250 series (2.4s)
Completed 150/250 series (2.4s)
Completed 160/250 series (2.4s)
Completed 170/250 series (2.4s)
Completed 180/250 series (2.4s)
Completed 190/250 series (2.4s)
Completed 200/250 series (2.4s)
Completed 210/250 series (2.4s)
Completed 220/250 series (2.4s)
Completed 230/250 series (2.4s)
Completed 240/250 series (2.4s)
Completed 250/250 series (2.5s)
                                      AR(1)         AR(2)         AR(3)          AR(4)          AR(5)
Selector                                                                                             
Minim

## Table 2.2-style result matrix

Rows are selectors; columns are true AR orders. Each entry contains the exact
selection count / 50, followed by (underselections / overselections). Failed
fits, if any, are reported separately and remain in the denominator.


In [4]:
display(run["matrix"])
display(run["summary"])
print("Failed fits:", run["verification"]["failed_fits"])


Failed fits: 0


,AR(1),AR(2),AR(3),AR(4),AR(5)
Selector,,,,,
Minimum width + descent chain,32/50 (0/18),22/50 (4/24),18/50 (7/25),22/50 (10/18),18/50 (14/18)
BIC,45/50 (5/0),42/50 (8/0),46/50 (4/0),40/50 (10/0),42/50 (7/1)
PACF cutoff,33/50 (4/13),29/50 (5/16),35/50 (4/11),38/50 (4/8),29/50 (5/16)


,method,order,correct,under,over,failed,total
0,Minimum width + descent chain,1,32,0,18,0,50
1,BIC,1,45,5,0,0,50
2,PACF cutoff,1,33,4,13,0,50
3,Minimum width + descent chain,2,22,4,24,0,50
4,BIC,2,42,8,0,0,50
5,PACF cutoff,2,29,5,16,0,50
6,Minimum width + descent chain,3,18,7,25,0,50
7,BIC,3,46,4,0,0,50
8,PACF cutoff,3,35,4,11,0,50
9,Minimum width + descent chain,4,22,10,18,0,50


## Verification and saved artifacts

Each original series is checked against its saved checksum. Per-series NPZ
checkpoints retain the fitted unit directions, widths, cloud diameters,
evaluated-state counts, BIC criteria, PACF values, and descent-chain scores.
The input archive is checked again after the experiment.

`selections.csv`, `summary.csv`, `width_curves.csv`, `descent_chain_scores.csv`,
and `failed_fits.csv` retain the results. `table_2_2_style.csv`,
`table_2_2_style.md`, and `table_2_2_style.tex` export the final matrix.
`configuration.json` records the candidate ranges, settings and source hashes;
`verification.json` records numerical checks and checkpoint checksums.


In [5]:
checks = run["verification"]
display(pd.Series({key: value for key, value in checks.items()
                   if key != "checkpoint_sha256"}, name="Verification"))
assert checks["input_files_still_match_manifest"]
if checks["failed_fits"]:
    display(results.loc[results.status.ne("ok"),
                        ["sample_id", "source_sample_id", "order", "method", "error_message"]])
print("CSV matrix:", run["directory"] / "table_2_2_style.csv")
print("LaTeX table:", run["directory"] / "table_2_2_style.tex")


CSV matrix: /Users/iljasukhanov/MASTERARBEIT/Code/Code_AR_processes/outputs/csv/ar_order_selection_final/030cde6f724e3c30/7ac02863db38b8d0/table_2_2_style.csv
LaTeX table: /Users/iljasukhanov/MASTERARBEIT/Code/Code_AR_processes/outputs/csv/ar_order_selection_final/030cde6f724e3c30/7ac02863db38b8d0/table_2_2_style.tex


series                                  250
methods                                   3
selections                              750
failed_fits                               0
successful_width_curves                 250
width_values                           2500
maximum_width_reconstruction_error      0.0
maximum_unit_norm_error                 0.0
maximum_upward_step                     0.0
maximum_chain_stationarity_residual     0.0
input_files_still_match_manifest       True
Name: Verification, dtype: object